📌 CELDA 1 – INSTALAR LIBRERÍAS

In [ ]:
!apt-get update
!apt-get install -y poppler-utils
!apt-get install -y tesseract-ocr
!apt-get install -y tesseract-ocr-spa
!pip install pdfplumber pandas sqlalchemy tabulate pytesseract pdf2image pillow

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,863 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,580 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,599 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [3,966 kB]
Fetched 18.3 MB in 7s (2,758

📌 CELDA 2 – IMPORTS Y CONFIGURACIÓN

In [ ]:
import pdfplumber
import pandas as pd
import sqlite3
import re
from datetime import datetime
from tabulate import tabulate
import pytesseract
from pdf2image import convert_from_path
from PIL import Image


📌 CELDA 3 – SUBIR PDFs

In [ ]:
from google.colab import files

uploaded = files.upload()
pdf_files = list(uploaded.keys())

pdf_files


📌 CELDA 4  – NORMALIZADOR DE TEXTO (IMPORTANTE)

In [ ]:
def normalizar_texto(texto):
    texto = texto.upper()
    texto = texto.replace(",", "")
    texto = re.sub(r"\s+", " ", texto)
    return texto


📌 CELDA 5 – FUNCIÓN SEGURA DE BÚSQUEDA

In [ ]:
def buscar_float(pattern, texto, default=0.0):
    match = re.search(pattern, texto, re.IGNORECASE)
    if match:
        try:
            return float(match.group(1))
        except:
            return default
    return default


📌 CELDA 6 – FUNCIÓN PARA LEER TEXTO DEL PDF

In [ ]:
def leer_pdf(path):
    texto = ""

    # 1️⃣ Intento normal (pdfplumber)
    try:
        with pdfplumber.open(path) as pdf:
            for page in pdf.pages:
                t = page.extract_text()
                if t:
                    texto += t + "\n"
    except:
        pass

    # 2️⃣ Si no hay texto → OCR
    if len(texto.strip()) < 50:
        print("🔎 PDF sin texto detectado → usando OCR")
        images = convert_from_path(path, dpi=300)
        for img in images:
            texto += pytesseract.image_to_string(img, lang="spa") + "\n"

    return texto


📌 CELDA 7 – PARSER BANCO DEL AUSTRO

In [ ]:
def parse_austro(texto):
    texto = normalizar_texto(texto)

    saldo_anterior = buscar_float(
        r"SALDO\s+ANTERIOR\s+([\d.]+)", texto
    )

    pagos_creditos = buscar_float(
        r"RESUMEN\s+DE\s+PAGOS/CR[EÉ]DITOS\s+([\d.]+)", texto
    )

    consumos_debitos = buscar_float(
        r"CONSUMOS/DE[BÍI]TOS\s+([\d.]+)", texto
    )

    saldo_total = buscar_float(
        r"TOTAL\s+A\s+PAGAR\s+([\d.]+)", texto
    )

    pago_minimo = buscar_float(
        r"M[IÍ]NIMO\s+A\s+PAGAR\s+([\d.]+)", texto
    )

    interes_mes = buscar_float(
        r"INTER[EÉ]S\s+FINANCIAMIENTO\s+N/D\s+([\d.]+)", texto
    )

    # CUPOS
    cupo_autorizado = buscar_float(
        r"AUTORIZADO\s+([\d.]+)", texto
    )

    cupo_utilizado = buscar_float(
        r"UTILIZADO\s+([\d.]+)", texto
    )

    cupo_disponible = buscar_float(
        r"DISPONIBLE\s+(-?[\d.]+)", texto
    )

    extracupo = buscar_float(
        r"EXTRACUPO\s+([\d.]+)", texto
    )

    tasa_anual = buscar_float(
        r"TASA\s+DE\s+INTER[EÉ]S\s+NOMINAL\s+([\d.]+)", texto
    )

    # PERIODO
    periodo = re.search(
        r"(\d{2}-[A-Z]{3}-\d{4})\s+AL\s+(\d{2}-[A-Z]{3}-\d{4})",
        texto
    )

    return {
        "banco": "BANCO DEL AUSTRO",
        "periodo_inicio": periodo.group(1) if periodo else None,
        "periodo_fin": periodo.group(2) if periodo else None,

        "saldo_anterior": saldo_anterior,
        "pagos_creditos": pagos_creditos,
        "consumos_debitos": consumos_debitos,

        "saldo_total": saldo_total,
        "pago_minimo": pago_minimo,

        "interes_mes": interes_mes,
        "tasa_anual": tasa_anual,

        "cupo_autorizado": cupo_autorizado,
        "cupo_utilizado": cupo_utilizado,
        "cupo_disponible": cupo_disponible,
        "extracupo": extracupo
    }


📌 CELDA 8 – PARSER BANCO DEL PACÍFICO (BASE)

In [ ]:
def parse_pacifico(texto):
    texto = normalizar_texto(texto)

    interes = buscar_float(
        r"INTER[EÉ]S\s+([\d.]+)",
        texto
    )

    saldo_anterior = buscar_float(
        r"SALDO\s+ANTERIOR\s+([\d.]+)",
        texto
    )

    total_pagar = buscar_float(
        r"TOTAL\s+A\s+PAGAR\s+(?:USD|US\$)?\s*([\d.]+)",
        texto
    )

    pago_minimo = buscar_float(
        r"(?:PAGO\s+)?M[IÍ]NIMO\s+(?:USD|US\$)?\s*([\d.]+)",
        texto
    )

    if pago_minimo == 0 and total_pagar > 0:
        pago_minimo = round(total_pagar * 0.05, 2)

    tasa = 17.0  # promedio TC Pacífico Ecuador

    periodo = re.search(
        r"PER[IÍ]ODO\s+DE\s+CORTE\s+DESDE:\s*(\d{2}/[A-Z]{3}/\d{4})\s+HASTA\s*(\d{2}/[A-Z]{3}/\d{4})",
        texto
    )


    return {
        "banco": "BANCO DEL PACIFICO",
        "saldo_anterior": saldo_anterior,
        "pagos_creditos": 0,
        "consumos_debitos": 0,
        "saldo_total": total_pagar,
        "pago_minimo": pago_minimo,
        "interes_mes": interes,
        "tasa_anual": tasa,
        "periodo_inicio": periodo.group(1) if periodo else None,
        "periodo_fin": periodo.group(2) if periodo else None,
        "cupo_autorizado": 0,
        "cupo_utilizado": 0,
        "cupo_disponible": 0,
        "extracupo": 0
    }


📌 CELDA 9 – CREAR BASE DE DATOS SQLITE

In [ ]:
conn = sqlite3.connect("finanzas.db")
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS estados_cuenta (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    banco TEXT NOT NULL,

    periodo_inicio TEXT NOT NULL,
    periodo_fin TEXT NOT NULL,

    saldo_anterior REAL,
    pagos_creditos REAL,
    consumos_debitos REAL,

    saldo_total REAL,
    pago_minimo REAL,

    interes_mes REAL,
    tasa_anual REAL,

    cupo_autorizado REAL,
    cupo_utilizado REAL,
    cupo_disponible REAL,
    extracupo REAL,

    creado_en TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    UNIQUE (banco, periodo_inicio, periodo_fin)
)
""")
conn.commit()


📌 CELDA 10 – PROCESAR PDFs Y GUARDAR EN SQLITE

In [ ]:
for pdf in pdf_files:
    print(f"\n📄 Procesando: {pdf}")
    texto = leer_pdf(pdf)
    texto_norm = normalizar_texto(texto)

    if "AUSTRO" in texto_norm:
        data = parse_austro(texto)
    elif "PACIFICO" in texto_norm or "PACÍFICO" in texto_norm or "BPACIFICO" in texto_norm:
        data = parse_pacifico(texto)
    else:
        print(f"❌ No reconocido: {pdf}")
        continue

    print("➡️ Datos detectados:")
    for k, v in data.items():
        print(f"   {k}: {v}")

    cur.execute("""
        INSERT OR IGNORE INTO estados_cuenta (
            banco,
            periodo_inicio,
            periodo_fin,
            saldo_anterior,
            pagos_creditos,
            consumos_debitos,
            saldo_total,
            pago_minimo,
            interes_mes,
            tasa_anual,
            cupo_autorizado,
            cupo_utilizado,
            cupo_disponible,
            extracupo
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            data["banco"],
            data["periodo_inicio"],
            data["periodo_fin"],
            data["saldo_anterior"],
            data["pagos_creditos"],
            data["consumos_debitos"],
            data["saldo_total"],
            data["pago_minimo"],
            data["interes_mes"],
            data["tasa_anual"],
            data["cupo_autorizado"],
            data["cupo_utilizado"],
            data["cupo_disponible"],
            data["extracupo"]
        ))

conn.commit()
print("\n✅ Estados de cuenta guardados en SQLite")


📌 CELDA 11 – VER RESUMEN FINANCIERO

In [ ]:
df = pd.read_sql("SELECT * FROM estados_cuenta", conn)
print(tabulate(df, headers="keys", tablefmt="psql"))


📌 CELDA 🔥 12 – ¿CUÁNTO INTERÉS ESTÁS PAGANDO?

In [ ]:
df.groupby("banco")["interes_mes"].sum()


📌 CELDA 🔥 13 – SIMULADOR DE PAGOS (CLAVE)

In [ ]:
def simular_pago(saldo, tasa_anual, pago):
    tasa_m = tasa_anual / 12 / 100
    meses = 0
    interes_total = 0

    while saldo > 0 and meses < 600:
        interes = saldo * tasa_m
        saldo = saldo + interes - pago
        interes_total += interes
        meses += 1

    return meses, round(interes_total, 2)


📌 CELDA 🔥 14 – PRUEBA REAL (AUSTRO)

In [ ]:
row = df[df["banco"] == "BANCO DEL AUSTRO"].iloc[-1]

for pago in [row["pago_minimo"], 300, 600, 1000]:
    meses, interes = simular_pago(row["saldo_total"], row["tasa_anual"], pago)
    print(f"PAGO ${pago} → {meses} meses | INTERÉS ${interes}")
